# Koiwriter
Writes English word to Tsevhu (Koilang) and then Koiwrit

### Imports

In [1]:
import json
import pandas as pd
from pathlib import Path
from typing import List, Union
from utils.tokenization import rules_for_tokens
from utils.orientation import choose_orientation
from utils.drawing import create_drawing_XML

## Read the excel sheet with the dictionary

In [2]:
dictionary_pd = pd.read_excel(Path("koilang.xlsx"), sheet_name="Dictionary", usecols="A:E")
dictionary_pd

,Word,Pronunciation,POS,Meaning,Verb Class
0,'eujkae,ʔœʒke,v,rot away,4
1,'eujvha,ʔœʒβɑ,v,rot,3
2,'iis'en,ʔɪsʔɛn,v,ripple; reflect light (often from water); spre...,1
3,aekak,ekɑk,v,land on; slam/smack into; crash; hit hard enou...,4
4,aekkae,ekke,v,smack; hit hard enough to injure someone,4
...,...,...,...,...,...
2521,yrieun,əɾiœn,adj,loyal; faithful,NaN
2522,yue,juɛ,adj,dry,NaN
2523,yul,jul,adj,early,NaN
2524,yuvi,juvi,adj,dark grey (like rain clouds),color


## Choose a translation

In [3]:
english_to_koilang = True
word_to_translate = "sun"

In [4]:
results = None
if english_to_koilang:
    results = dictionary_pd.loc[dictionary_pd['Meaning'].str.contains(word_to_translate)]
else:
    # Saved for future where reverse translation is done
    results = dictionary_pd.loc[dictionary_pd['Word'].str.contains(word_to_translate)]

results.set_index("Word", inplace=True)
results.reset_index(inplace=True)
results.index += 1
num_of_results = len(results.index)
chosen_translation = None

if num_of_results == 0:
    print("No translations found.")

elif num_of_results == 1:
    print("Translation:")
    chosen_translation = results.iloc[[0]]
    
elif num_of_results > 1:
    print("Multiple options exist. Please choose one:")
    print(results, end="\n\n")
    
    # Allow for a choice
    valid_choice = False
    
    while not valid_choice:
        choice = input("Choice (1, 2, 3, ...): ")
        try:
            if choice.isdigit() and int(choice) in range(1, int(len(results.index)) + 1):
                choice = int(choice)
                chosen_translation = results.iloc[[choice - 1]]
                valid_choice = True
        except:
            print(f"Input is not a number or it is not in range {choice}")
    
    print(f"Chosen translation:")

print(chosen_translation)
        
    

Multiple options exist. Please choose one:
         Word Pronunciation POS  \
1     aesiivh          esɪβ   v   
2   kombaekae       kombeke   v   
3         nak           nɑk   v   
4     li'enak       liʔɛnɑk   n   
5       Oitje           NaN   n   
6        soem          soɛm   n   
7     soemrha       soɛmr̥ɑ   n   
8     soemsii        soɛmsɪ   n   
9    soemsyun      soɛmsʲun   n   
10       syun          sjun   n   

                                              Meaning Verb Class  
1   sink; drown; go below; slip under; under(groun...          3  
2   divide apart; split apart; divide asunder (imp...          4  
3                        sunbathe; sit out in the sun          1  
4   lizard (lit. sunbathing shield (shield often r...     animal  
5   earth deity (connected to a myth of the sun in...        NaN  
6                                                 sun        NaN  
7                      sunrise (can shorten to sorha)        NaN  
8                       sunset (can

## Separate word into writeable parts

In [5]:
tokens = rules_for_tokens(chosen_translation["Word"].values[0])
# tokens = rules_for_tokens("Tsevhu")
tokens

['l', 'i', '`', 'e', 'n', 'a', 'k']

## Determine the best orientation for each ripple

In [6]:
# Get density data
with open(Path("ripples/density_data.json")) as f:
    density_data = json.loads(f.read())

In [7]:
# Get how to translate the orientations
with open(Path("ripples/translate_orientation.json")) as f:
    translate_orientation = json.loads(f.read())

In [8]:
# Choose orientations based on densities
orientations = choose_orientation(tokens, density_data, translate_orientation)
orientations
    
    

[0, 3, 0, 1, 0, 2, 2]

## Draw the ripples

In [ ]:
create_drawing_XML(tokens, orientations, path_to_ripple_images=Path('./ripples/images'), scale=0.8, stroke_width=6).write('output.svg')